<a href="https://colab.research.google.com/github/himanshusar123/-Machine-Learning-Quiz-Classification-or-Regression-/blob/main/Day_4_Harbinger_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Day 4 – RetailMax HR Analytics System Version 4.0
## Theme: "From Data Storage to Business Intelligence"

### Story Continuation
#### 9:00 AM – Morning Briefing
You are the CTO. Yesterday, you successfully migrated the corporate HR records from a flat CSV file into a database. The HR Director is extremely pleased. But this morning, the CEO calls a meeting:

> "Team, RetailMax is growing to 500 stores and 15,000 employees. I don't want to look at 15,000 raw database rows. I need to make business decisions: Where should we hire? Which departments have the highest salary costs? I need reports, trends, charts, and KPIs. Bring me insights, not raw data."

#### The BI Value Chain
```text
Raw Data (CSV/DB) ──► Information (KPIs/Stats) ──► Insights (Trends/Charts) ──► Business Decisions
```

Today, we transition from software developers to **data analysts** and introduce **Pandas** for high-performance data analytics.

### Setup & Data Generation
To simulate yesterday's SQLite database and our dirty source datasets, let's run this setup code to generate our data files directly inside our environment.

In [ ]:
# Data Generation Code
import csv
import sqlite3
import random
import os
from datetime import datetime, timedelta

first_names = ["John", "Jane", "Alice", "Bob", "Charlie", "Diana", "Ethan", "Fiona", "George", "Hannah"]
last_names = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Miller", "Davis", "Garcia"]
departments = ["HR", "Engineering", "Finance", "Sales", "Marketing"]

# Generate clean CSV
random.seed(42)
clean_employees = []
for emp_id in range(1001, 1101):
    dept = random.choice(departments)
    name = f"{random.choice(first_names)} {random.choice(last_names)}"
    salary = random.randint(45000, 120000)
    clean_employees.append({
        "EmployeeID": emp_id,
        "Name": name,
        "Department": dept,
        "Salary": salary,
        "JoiningDate": "2023-05-15",
        "PerformanceScore": random.randint(1, 5)
    })

with open("employees.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=clean_employees[0].keys())
    writer.writeheader()
    writer.writerows(clean_employees)

# Generate SQLite database
conn = sqlite3.connect("employees.db")
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS employees")
cursor.execute("""
    CREATE TABLE employees (
        EmployeeID INTEGER PRIMARY KEY,
        Name TEXT,
        Department TEXT,
        Salary INTEGER,
        JoiningDate TEXT,
        PerformanceScore INTEGER
    )
""")
for emp in clean_employees:
    cursor.execute("INSERT INTO employees VALUES (?,?,?,?,?,?)", 
                   (emp["EmployeeID"], emp["Name"], emp["Department"], emp["Salary"], emp["JoiningDate"], emp["PerformanceScore"]))
conn.commit()
conn.close()

# Generate dirty CSV
dirty_employees = [clean_employees[i].copy() for i in range(20)]
# Add duplicate
dirty_employees.append(dirty_employees[0].copy())
# Add missing salary
dirty_employees[1]["Salary"] = ""
# Add negative salary
dirty_employees[2]["Salary"] = -55000
# Add invalid department
dirty_employees[3]["Department"] = "Financcce"

with open("employees_dirty.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=dirty_employees[0].keys())
    writer.writeheader()
    writer.writerows(dirty_employees)

print("Setup completed! Data files generated successfully.")

## Sprint 1 – Loading Data
Learn how to load data from standard CSV files and databases using Pandas.

In [ ]:
import pandas as pd
import sqlite3

# 1. Load CSV data
df_csv = pd.read_csv("employees.csv")
print("Loaded CSV. Columns:", df_csv.columns)

# 2. Load SQLite database data
conn = sqlite3.connect("employees.db")
df = pd.read_sql_query("SELECT * FROM employees", conn)
conn.close()
print("Loaded SQLite DB. Shape:", df.shape)

## Sprint 2 – Explore Data
Before performing analytics, inspect the structure of the loaded DataFrame.

In [ ]:
print("--- First 5 Rows ---")
print(df.head())

print("\n--- DataFrame Info ---")
df.info()

print("\n--- Descriptive Statistics ---")
print(df.describe())

## Sprint 3 – Data Cleaning
Clean the dirty records by identifying duplicates, handling missing salaries, and resetting indices.

In [ ]:
df_dirty = pd.read_csv("employees_dirty.csv")

print("Total duplicates found:", df_dirty.duplicated(subset=["EmployeeID"]).sum())
print("Total missing salaries found:", df_dirty["Salary"].isnull().sum())

# Clean missing and duplicate values
df_clean = df_dirty.drop_duplicates(subset=["EmployeeID"], keep="first").copy()
median_salary = pd.to_numeric(df_clean["Salary"], errors="coerce").median()
df_clean["Salary"] = df_clean["Salary"].fillna(median_salary)

print("\nCleaned data shape:", df_clean.shape)

## Sprint 4 & 5 – HR Analytics and Filtering
Compute aggregated metrics and run conditional filtering queries.

In [ ]:
# SPRINT 4: Aggregations
print(f"Total Employees: {len(df)}")
print(f"Average Salary: ${df['Salary'].mean():,.2f}")
print(f"Highest Salary: ${df['Salary'].max():,.2f}")
print(f"Lowest Salary: ${df['Salary'].min():,.2f}")

print("\nHeadcount per Department:")
print(df.groupby("Department").size())

# SPRINT 5: Filtering
finance_employees = df[df["Department"] == "Finance"]
print(f"\nEmployees in Finance: {len(finance_employees)}")

high_earning_eng = df[(df["Department"] == "Engineering") & (df["Salary"] > 80000)]
print(f"High-paid Engineers: {len(high_earning_eng)}")

## Sprint 6 – Sorting
Identify top paid employees.

In [ ]:
print("Top 5 highest paid employees:")
print(df.sort_values("Salary", ascending=False).head(5)[["Name", "Department", "Salary"]])

print("\nUsing df.nlargest:")
print(df.nlargest(3, "Salary")[["Name", "Department", "Salary"]])

## Sprint 7 & 8 – Visualizing and Exporting Reports
Convert analytical tables into visual charts and export reports to CSV or Excel.

In [ ]:
import matplotlib.pyplot as plt

# Sprint 7: Matplotlib Visualizations
plt.figure(figsize=(6, 3))
df["Department"].value_counts().plot(kind="bar", color="lightgreen", edgecolor="black")
plt.title("Employee Counts by Department")
plt.ylabel("Count")
plt.xlabel("Department")
plt.show()

# Sprint 8: Exporting reports
df.to_csv("final_analytics_report.csv", index=False)
print("Report exported to CSV successfully!")

## 🚀 Special Topics
### 1. Data Tool Comparison (Pandas vs. DuckDB)
Let's see how DuckDB can query our in-memory Pandas DataFrame directly using standard SQL.

In [ ]:
import duckdb

print("--- DuckDB SQL Result ---")
duck_df = duckdb.query("""
    SELECT Department, AVG(Salary) as AvgSalary, COUNT(*) as Headcount
    FROM df
    GROUP BY Department
""").to_df()
print(duck_df)

### 2. Data Contracts
Verify incoming data against schema rules to catch corrupt entries (such as negative salaries or wrong departments).

In [ ]:
valid_depts = ["HR", "Engineering", "Finance", "Sales", "Marketing"]
raw_data = pd.read_csv("employees_dirty.csv")

print("Checking Data Quality Invariants...")

# Rule 1: Missing IDs
missing_ids = raw_data["EmployeeID"].isnull().sum()
if missing_ids > 0:
    print(f"-> [FAIL] Missing IDs found: {missing_ids} record(s)")

# Rule 2: Negative Salary
raw_data["Salary"] = pd.to_numeric(raw_data["Salary"], errors="coerce")
neg_salaries = (raw_data["Salary"] < 0).sum()
if neg_salaries > 0:
    print(f"-> [FAIL] Negative salaries found: {neg_salaries} record(s)")
    print(raw_data[raw_data["Salary"] < 0][["Name", "Salary"]])

# Rule 3: Invalid Departments
invalid_depts = (~raw_data["Department"].isin(valid_depts)).sum()
if invalid_depts > 0:
    print(f"-> [FAIL] Invalid departments found: {invalid_depts} record(s)")
    print(raw_data[~raw_data["Department"].isin(valid_depts)][["Name", "Department"]])